In [1]:
#pip install ollama==0.4.7
#ollama run llama3.1
import ollama
llm = "llama3.1"

In [2]:
stream = ollama.generate(model=llm, prompt='''what time is it?''', stream=True)
for chunk in stream:
    print(chunk['response'], end='', flush=True)

However, I'm a large language model, I don't have real-time access to your location or current time. But I can suggest a few options:

1. Check your device: You can check the time on your phone, watch, or computer.
2. Ask a voice assistant: If you have a smart speaker or virtual assistant like Siri, Google Assistant, or Alexa, you can ask them for the current time.
3. Look at an online clock: There are many websites that display the current time in different parts of the world.

If you want to get an estimate of the current time based on your location, I can try to help with that. Can you please tell me what city or region you're in?

In [4]:
#pip install langchain-community==0.3.17
#pip install duckduckgo_search==6.3.5
from langchain_community.tools import DuckDuckGoSearchResults

def search_web(query: str) -> str:
    return DuckDuckGoSearchResults(backend="news").run(query)

tool_search_web = {'type':'function', 'function':{
    'name': 'search_web',
    'description': 'Search the web',
    'parameters': {'type': 'object',
                   'required': ['query'],
                   'properties': {
                       'query': {'type': 'str', 'description': 'the topic or subject to search on the web'},
}}}}

search_web(query='nvidia')

"snippet: Nvidia CEO Jensen Huang on Thursday walked back comments he made in January, when he cast doubt on whether useful quantum computers would hit the market in the next 15 years., title: Nvidia CEO Huang says he was wrong about timeline for quantum, surprised his comments hurt stocks, link: https://www.msn.com/en-us/news/technology/nvidia-ceo-huang-says-he-was-wrong-about-timeline-for-quantum-surprised-his-comments-hurt-stocks/ar-AA1BoRf0, date: 2025-03-21T16:23:57+00:00, source: NBC News, snippet: But his latest foray into the quantum world appeared to have the opposite effect. Despite Huang's public change of tune, several key stocks in the sector tumbled on Thursday, with, title: Nvidia's CEO sought to help the quantum companies and ended up causing another sell-off in the stocks, link: https://www.msn.com/en-us/technology/tech-companies/nvidia-s-ceo-sought-to-help-the-quantum-companies-and-ended-up-causing-another-sell-off-in-the-stocks/ar-AA1BoMLv, date: 2025-03-21T16:18:34+

In [5]:
def search_yf(query: str) -> str:
    return DuckDuckGoSearchResults(backend="news").run(f"site:finance.yahoo.com {query}")

tool_search_yf = {'type':'function', 'function':{
    'name': 'search_yf',
    'description': 'Search for specific financial news',
    'parameters': {'type': 'object',
                   'required': ['query'],
                   'properties': {
                       'query': {'type': 'str', 'description': 'the financial topic or subject to search'},
}}}}

search_web(query='nvidia')

"snippet: Nvidia CEO Jensen Huang on Thursday walked back comments he made in January, when he cast doubt on whether useful quantum computers would hit the market in the next 15 years., title: Nvidia CEO Huang says he was wrong about timeline for quantum, surprised his comments hurt stocks, link: https://www.msn.com/en-us/news/technology/nvidia-ceo-huang-says-he-was-wrong-about-timeline-for-quantum-surprised-his-comments-hurt-stocks/ar-AA1BoRf0, date: 2025-03-21T16:23:57+00:00, source: NBC News, snippet: In theory, a quantum computer could perform calculations vastly more quickly than today's conventional computers., title: Boston's hot quantum computing scene reels in Nvidia, the biggest name in tech, link: https://www.bostonglobe.com/2025/03/21/business/nvidia-boston-quantum-computing/, date: 2025-03-21T18:13:00+00:00, source: The Boston Globe, snippet: The fashion at Nvidia's GTC 2025 conference was featured audiences full of half zip sweaters, Hoka tennis shoes, and puffer jackets.

In [6]:
prompt = '''You are an assistant with access to tools, you must decide when to use tools to answer user message.'''
messages = [{"role":"system", "content":prompt}]

while True:
    ## user input
    try:
        q = input('🙂 >')
    except EOFError:
        break
    if q == "quit":
        break
    if q.strip() == "":
        continue
    messages.append( {"role":"user", "content":q} )

    ## model
    agent_res = ollama.chat(
        model=llm,
        # format="json",  # or schema
        stream=False,
        tools=[tool_search_web, tool_search_yf],
        messages=messages)

    ## response
    dic_tools = {'search_web':search_web, 'search_yf':search_yf}

    # print( dir( agent_res["message"] ) )

    # if "tool_calls" in agent_res["message"].keys():
    if "tool_calls" in dir( agent_res["message"] ):
        for tool in agent_res["message"]["tool_calls"]:
            t_name, t_inputs = tool["function"]["name"], tool["function"]["arguments"]
            
            if f := dic_tools.get(t_name):
                ### calling tool
                print('🔧 >', f"\x1b[1;31m{t_name} -> Inputs: {t_inputs}\x1b[0m")
                messages.append( {"role":"user", "content":"use tool '"+t_name+"' with inputs: "+str(t_inputs)} )

                ### tool output
                t_output = f(**tool["function"]["arguments"])
                print(t_output)

                ### final res
                p = f'''Summarize this to answer user qustion, be as concise as possible: {t_output}'''
                res = ollama.generate(model=llm, prompt=q+". "+p)["response"]
            else:
                print('🤬 >', f"\x1b[1;31m{t_name} -> NotFound\x1b[0m")

    if agent_res['message']['content'] != '':
        res = agent_res['message']['content']

    print("👽 >", f"\x1b[1;30m{res}\x1b[0m")
    messages.append( {"role":"assistant", "content":res} )

🙂 > Tell me about Powell


🔧 > search_web -> Inputs: {'query': 'Powell'}
snippet: Federal Reserve Chair Jerome Powell has been crystal clear about whether President Donald Trump can fire him. "Not permitted under the law," he said — twice — when asked in November., title: Jerome Powell Says Trump Can't Fire Him. That Might Change, link: https://www.msn.com/en-us/money/markets/jerome-powell-says-trump-can-t-fire-him-that-might-change/ar-AA1BnCtr, date: 2025-03-21T11:00:00+00:00, source: Bloomberg, snippet: It's too early to say if "the inflationary effects will be transitory, especially given that companies and households still have fresh in their minds the recent history of high unanticipated inflation," Mohamed El-Erian, the economist who called the Fed too cautious in handling the last inflation surge, wrote on X., title: Powell Revives a Debate on 'Transitory' Inflation, link: https://www.nytimes.com/2025/03/20/business/dealbook/powell-tariff-transitory-inflation.html, date: 2025-03-20T15:51:00+00:00, source:

🙂 > be more specific about the rate cut


🔧 > search_web -> Inputs: {'query': 'Powell rate cut explanation'}
snippet: Prices are now projected to rise faster than expected at least in part and perhaps largely due to Trump's plans to levy duties on imports from U.S. trading partners, Fed Chair Powell said., title: Fed in no rush to cut rates; Trump disagrees, link: https://www.msn.com/en-us/money/markets/fed-in-no-rush-to-cut-rates-trump-disagrees/ar-AA1BkvFn, date: 2025-03-20T18:12:12+00:00, source: Reuters, snippet: Federal Reserve Chair Jerome Powell hinted the central bank is unlikely to deliver President Trump his much-desired interest rate cuts anytime soon - citing uncertainty around the president's own trade policies as cause for the delay., title: Fed chair Jerome Powell says he's in no hurry to cut rates amid Trump policy shifts, link: https://www.msn.com/en-us/money/markets/fed-chair-jerome-powell-says-he-s-in-no-hurry-to-cut-rates-amid-trump-policy-shifts/ar-AA1Atl0N, date: 2025-03-07T18:18:00+00:00, source: New Yor

🙂 > quit


In [7]:
messages

[{'role': 'system',
  'content': 'You are an assistant with access to tools, you must decide when to use tools to answer user message.'},
 {'role': 'user', 'content': 'Tell me about Powell'},
 {'role': 'user',
  'content': "use tool 'search_web' with inputs: {'query': 'Powell'}"},
 {'role': 'assistant',
  'content': "Here is a concise answer to the user's question:\n\nFederal Reserve Chair Jerome Powell has stated that President Donald Trump cannot fire him under the law."},
 {'role': 'user', 'content': 'be more specific about the rate cut'},
 {'role': 'user',
  'content': "use tool 'search_web' with inputs: {'query': 'Powell rate cut explanation'}"},
 {'role': 'assistant',
  'content': "The Fed is not planning to cut interest rates due to uncertainty caused by Trump's trade policies, which may lead to higher inflation. There are no specific rate cuts announced yet, but the Fed may signal a possible cut in its statement."}]

In [8]:
agent_res

ChatResponse(model='llama3.1', created_at='2025-03-22T01:06:27.930068295Z', done=True, done_reason='stop', total_duration=6773548337, load_duration=19553500, prompt_eval_count=299, prompt_eval_duration=5694000000, eval_count=21, eval_duration=1057000000, message=Message(role='assistant', content='', images=None, tool_calls=[ToolCall(function=Function(name='search_web', arguments={'query': 'Powell rate cut explanation'}))]))